# EDA bộ dữ liệu `raw.csv`

Notebook này khảo sát chất lượng và đặc điểm ban đầu của dữ liệu sau bước
merge. Kết quả EDA được dùng để phát hiện dữ liệu rỗng, trùng lặp, sai lệch
giữa các nền tảng và các đặc điểm ngôn ngữ cần lưu ý trước khi xử lý tiếp.

## Phạm vi và giới hạn của dữ liệu hiện tại

`raw.csv` hiện có bốn trường: `row_id`, `platform`, `title` và `comment`.
Do raw không giữ `post_id`, `comment_id` hoặc trường thời gian, notebook sẽ:

- dùng số tiêu đề duy nhất theo nền tảng như một **ước lượng số bài đăng**;
- không suy đoán ngày, giờ hoặc tuần từ `row_id`;
- ghi rõ khi không thể tính phân bố thời gian;
- đếm URL, hashtag, mention và emoji trên nội dung `title` kết hợp với
  `comment`;
- dùng nhận diện ngôn ngữ dạng heuristic, nên kết quả cần được xem là
  thống kê thăm dò chứ không phải nhãn ngôn ngữ tuyệt đối.

## 1. Nạp thư viện và cấu hình

In [ ]:
from pathlib import Path
import re
import unicodedata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 120)
plt.style.use('seaborn-v0_8-whitegrid')

CANDIDATE_PATHS = [
]
DATA_PATH = next((path for path in CANDIDATE_PATHS if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError('Không tìm thấy raw.csv. Hãy chạy notebook từ DS108 hoặc Preprocessing.')

print(f'Đọc dữ liệu từ: {DATA_PATH.resolve()}')

## 2. Đọc dữ liệu và kiểm tra schema

In [ ]:
df = pd.read_csv(DATA_PATH, dtype='string', keep_default_na=False)
df = df.fillna('')

EXPECTED_COLUMNS = {'row_id', 'platform', 'title', 'comment'}
missing_columns = EXPECTED_COLUMNS - set(df.columns)
if missing_columns:
    raise ValueError(f'Thiếu cột bắt buộc: {sorted(missing_columns)}')

for column in ['platform', 'title', 'comment']:
    df[column] = df[column].astype('string').fillna('')

print(f'Số bản ghi: {len(df):,}')
print(f'Số cột: {len(df.columns)}')
display(df.head())
display(df.dtypes.to_frame('dtype'))

## 3. Tổng số bài đăng, bình luận và dữ liệu theo nền tảng

Raw không có `post_id` hoặc `record_type`, nên không thể xác định chính xác
số bài đăng. Số tiêu đề duy nhất chỉ được dùng làm ước lượng tham khảo.
Tổng số bình luận được tính là số dòng có nội dung trong trường `comment`.

In [ ]:
df['platform'] = df['platform'].str.strip().replace('', 'unknown')
df['title_nonempty'] = df['title'].str.strip().ne('')
df['comment_nonempty'] = df['comment'].str.strip().ne('')

total_records = len(df)
total_comments = int(df['comment_nonempty'].sum())
post_proxy = (
    df.loc[df['title_nonempty'], ['platform', 'title']]
      .drop_duplicates()
).shape[0]

summary_counts = pd.DataFrame({
    'Chỉ số': ['Tổng số bản ghi', 'Tổng số bình luận có nội dung',
              'Số tiêu đề duy nhất - proxy cho bài đăng'],
    'Giá trị': [total_records, total_comments, post_proxy],
    'Ghi chú': [
        'Tất cả dòng trong raw.csv',
        'comment sau khi bỏ khoảng trắng đầu/cuối',
        'Không phải số bài đăng chính xác vì raw không có post_id',
    ],
})
display(summary_counts)

platform_counts = (
    df['platform'].value_counts(dropna=False)
      .rename_axis('platform')
      .reset_index(name='records')
)
platform_counts['share'] = platform_counts['records'] / total_records
display(platform_counts.style.format({'records': '{:,}', 'share': '{:.2%}'}))

ax = platform_counts.plot.bar(x='platform', y='records', legend=False, figsize=(9, 4), color='#2878b5')
ax.set_title('Số lượng bản ghi theo nền tảng')
ax.set_xlabel('Nền tảng')
ax.set_ylabel('Số bản ghi')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 4. Dữ liệu rỗng và bản ghi trùng lặp

In [ ]:
empty_summary = pd.DataFrame({
    'Trường': ['title', 'comment', 'title hoặc comment'],
    'Số dòng rỗng': [
        int((~df['title_nonempty']).sum()),
        int((~df['comment_nonempty']).sum()),
        int((~(df['title_nonempty'] & df['comment_nonempty'])).sum()),
    ],
})
empty_summary['Tỷ lệ'] = empty_summary['Số dòng rỗng'] / total_records
display(empty_summary.style.format({'Số dòng rỗng': '{:,}', 'Tỷ lệ': '{:.2%}'}))

# Không dùng row_id để tìm trùng vì row_id được tạo lại khi merge.
dedup_key = (
    df[['platform', 'title', 'comment']]
      .astype('string')
      .apply(lambda column: column.str.strip().str.casefold())
)
duplicate_mask = dedup_key.duplicated(keep=False)
duplicate_rows = int(duplicate_mask.sum())
duplicate_groups = int(dedup_key.loc[duplicate_mask].drop_duplicates().shape[0])

duplicate_summary = pd.DataFrame({
    'Chỉ số': ['Bản ghi thuộc nhóm trùng', 'Số nhóm trùng', 'Tỷ lệ bản ghi thuộc nhóm trùng'],
    'Giá trị': [duplicate_rows, duplicate_groups, duplicate_rows / total_records],
})
display(duplicate_summary.style.format({'Giá trị': lambda value: f'{value:.2%}' if isinstance(value, float) else f'{value:,}'}))

print('Lưu ý: giống nội dung không đồng nghĩa chắc chắn là cùng một bình luận; cần xem xét platform và title trước khi loại.')

## 5. Độ dài `title` và `comment`

Độ dài được tính theo số ký tự sau khi bỏ khoảng trắng đầu và cuối. Thống kê
cả trung bình, trung vị và các phân vị giúp tránh việc chỉ nhìn vào giá trị
trung bình bị kéo lệch bởi các bình luận rất dài.

In [ ]:
df['title_chars'] = df['title'].str.strip().str.len()
df['comment_chars'] = df['comment'].str.strip().str.len()
df['title_words'] = df['title'].str.strip().str.split().str.len()
df['comment_words'] = df['comment'].str.strip().str.split().str.len()

length_summary = pd.DataFrame({
    'Trường': ['title', 'comment'],
    'Trung bình ký tự': [df['title_chars'].mean(), df['comment_chars'].mean()],
    'Trung vị ký tự': [df['title_chars'].median(), df['comment_chars'].median()],
    'P25': [df['title_chars'].quantile(.25), df['comment_chars'].quantile(.25)],
    'P75': [df['title_chars'].quantile(.75), df['comment_chars'].quantile(.75)],
    'Lớn nhất': [df['title_chars'].max(), df['comment_chars'].max()],
})
display(length_summary.style.format({
    'Trung bình ký tự': '{:.2f}', 'Trung vị ký tự': '{:.0f}',
    'P25': '{:.0f}', 'P75': '{:.0f}', 'Lớn nhất': '{:,.0f}'
}))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df.loc[df['title_chars'] > 0, 'title_chars'].plot.hist(bins=40, ax=axes[0], color='#55a868')
df.loc[df['comment_chars'] > 0, 'comment_chars'].plot.hist(bins=40, ax=axes[1], color='#c44e52')
axes[0].set_title('Phân bố độ dài title')
axes[1].set_title('Phân bố độ dài comment')
for ax in axes:
    ax.set_xlabel('Số ký tự')
    ax.set_ylabel('Số bản ghi')
plt.tight_layout()
plt.show()

## 7. Ước lượng tỷ lệ ngôn ngữ

Đây là bộ phân loại heuristic phục vụ EDA, gồm tiếng Việt chủ yếu, tiếng Anh
chủ yếu, ngôn ngữ hỗn hợp, chỉ emoji/ký hiệu và không xác định. Kết quả này
không thay thế một bộ nhận diện ngôn ngữ chuyên dụng.

In [ ]:
VI_DIACRITICS = re.compile(r'[ăâđêôơưáàảãạấầẩẫậắằẳẵặéèẻẽẹếềểễệíìỉĩịóòỏõọốồổỗộớờởỡợúùủũụứừửữựýỳỷỹỵ]', re.IGNORECASE)
VI_WORDS = re.compile(r'\b(anh|chị|em|tôi|mình|bạn|không|được|với|này|kia|rất|quá|nhưng|và|là|có|cho|của|một|những|người|thích|ghét|đẹp|xấu)\b', re.IGNORECASE)
EN_WORDS = re.compile(r'\b(the|and|you|your|this|that|with|for|not|very|good|bad|love|hate|like|comment|please|thanks)\b', re.IGNORECASE)
LETTER_RE = re.compile(r'[A-Za-zÀ-ỹ]')

def language_group(text: str) -> str:
    text = str(text).strip()
    if not text:
        return 'empty'
    has_letters = bool(LETTER_RE.search(text))
    if not has_letters:
        return 'emoji_or_symbol'
    has_vi = bool(VI_DIACRITICS.search(text) or VI_WORDS.search(text))
    has_en = bool(EN_WORDS.search(text))
    if has_vi and has_en:
        return 'mixed'
    if has_vi:
        return 'vi_primary'
    if has_en:
        return 'en_primary'
    return 'unknown'

df['language_group'] = (df['title'].fillna('') + ' ' + df['comment'].fillna('')).map(language_group)
language_counts = (
    df['language_group'].value_counts()
      .rename_axis('language_group')
      .reset_index(name='records')
)
language_counts['share'] = language_counts['records'] / total_records
display(language_counts.style.format({'records': '{:,}', 'share': '{:.2%}'}))

ax = language_counts.plot.bar(x='language_group', y='share', legend=False, figsize=(9, 4), color='#8172b2')
ax.set_title('Tỷ lệ nhóm ngôn ngữ - heuristic')
ax.set_xlabel('Nhóm ngôn ngữ')
ax.set_ylabel('Tỷ lệ')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda value, _: f'{value:.0%}'))
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 8. URL, hashtag, mention và emoji

Báo cáo cả số dòng có chứa đặc trưng và tổng số lần xuất hiện. Mention dạng
`@tên_người_dùng` được thống kê để kiểm tra dữ liệu còn sót, không xem đó là
nội dung có giá trị ngữ nghĩa. Các emoji cần bảo toàn gồm `✅`, `❤`, `😂`,
`😭`, `🥰`, `🤣` và `👍`.

In [ ]:
text_all = (df['title'].fillna('') + ' ' + df['comment'].fillna('')).astype('string')
URL_RE = r'https?://\S+|www\.\S+'
HASHTAG_RE = r'(?<!\w)#\w+'
MENTION_RE = r'(?<![\w.])@[A-Za-z0-9_][A-Za-z0-9_.-]*'
EMOJI_RE = re.compile(r'[\U0001F300-\U0001FAFF\u2600-\u27BF]')
PROTECTED_EMOJIS = ['✅', '❤', '😂', '😭', '🥰', '🤣', '👍']

feature_rows = []
for name, pattern in [('URL', URL_RE), ('hashtag', HASHTAG_RE), ('mention', MENTION_RE)]:
    occurrences = text_all.str.count(pattern)
    feature_rows.append({
        'feature': name,
        'rows_containing': int(occurrences.gt(0).sum()),
        'occurrences': int(occurrences.sum()),
    })

emoji_occurrences = text_all.map(lambda value: len(EMOJI_RE.findall(str(value))))
feature_rows.append({
    'feature': 'emoji',
    'rows_containing': int(emoji_occurrences.gt(0).sum()),
    'occurrences': int(emoji_occurrences.sum()),
})
feature_summary = pd.DataFrame(feature_rows)
display(feature_summary.style.format({'rows_containing': '{:,}', 'occurrences': '{:,}'}))

protected_rows = {emoji: int(text_all.str.contains(re.escape(emoji), regex=True).sum()) for emoji in PROTECTED_EMOJIS}
protected_occurrences = {emoji: int(text_all.str.count(re.escape(emoji)).sum()) for emoji in PROTECTED_EMOJIS}
protected_summary = pd.DataFrame({
    'emoji': PROTECTED_EMOJIS,
    'rows_containing': [protected_rows[e] for e in PROTECTED_EMOJIS],
    'occurrences': [protected_occurrences[e] for e in PROTECTED_EMOJIS],
})
display(protected_summary)

ax = feature_summary.plot.bar(x='feature', y='rows_containing', legend=False, figsize=(9, 4), color='#dd8452')
ax.set_title('Số dòng chứa URL, hashtag, mention và emoji')
ax.set_xlabel('Đặc trưng')
ax.set_ylabel('Số dòng')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 10. Tổng hợp kết luận EDA

Sau khi chạy toàn bộ notebook, cần ghi lại các phát hiện chính:

- Nền tảng nào chiếm tỷ trọng lớn nhất và dữ liệu có mất cân bằng theo nguồn
  hay không.
- Tỷ lệ rỗng và tỷ lệ trùng lặp có đáng kể hay không.
- Bình luận thường ngắn hay dài; có nhiều bình luận chỉ gồm emoji hay không.
- Tỷ lệ URL, hashtag, mention và emoji còn sót lại như thế nào.
- Heuristic ngôn ngữ cho thấy nhóm tiếng Việt, tiếng Anh, hỗn hợp và không
  xác định phân bố ra sao.
- Có thể phân tích theo thời gian hay cần bổ sung trường datetime từ metadata
  nội bộ trước khi merge.

Các kết quả EDA chỉ mô tả hiện trạng dữ liệu; không tự động quyết định bản ghi
nào được giữ hoặc loại nếu chưa có quy tắc tiền xử lý tương ứng.